# 🏥 Diagnostic Médical IA — BioMedLM Fine-tuning
### Notebook 3 : Inférence, Évaluation & Interface

**Ce notebook :**
- Charge le modèle fine-tuné
- Effectue des tests d'inférence sur de nouveaux cas
- Évalue les performances (ROUGE, BLEU)
- Lance une interface Gradio interactive

In [ ]:
!pip install -q gradio
print('✅ Gradio installé')

In [ ]:
# ============================================================
# CELLULE 1 — Chargement du modèle fine-tuné
# ============================================================
import torch
import warnings
warnings.filterwarnings('ignore')

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

MODEL_PATH = '/content/medical_ai/model_finetuned'
BASE_MODEL  = 'stanford-crfm/BioMedLM'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

print('⏳ Chargement du tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('⏳ Chargement du modèle fine-tuné...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, MODEL_PATH)
model.eval()

print('✅ Modèle prêt pour l\'inférence !')

In [ ]:
# ============================================================
# CELLULE 2 — Fonction d'inférence principale
# ============================================================

SYSTEM_PROMPT = """Tu es MediAI, un assistant médical expert formé pour aider les professionnels de santé.
Tu analyses les symptômes présentés, proposes des diagnostics différentiels, et formules des recommandations.
IMPORTANT : Tes analyses sont des outils d'aide à la décision. Seul un médecin peut établir un diagnostic officiel.
En cas d'urgence vitale, oriente toujours vers les services d'urgence (15, 18, 112)."""

def diagnose(
    symptoms: str,
    patient_info: str = '',
    max_new_tokens: int = 600,
    temperature: float = 0.3,
    top_p: float = 0.9
) -> str:
    """
    Génère un diagnostic médical à partir des symptômes.
    
    Args:
        symptoms    : Description des symptômes
        patient_info: Informations patient (âge, sexe, antécédents)
        max_new_tokens: Longueur max de la réponse
        temperature : Créativité (0.1=conservateur, 0.7=créatif)
        top_p       : Nucleus sampling
    """
    patient_str = f'\nPatient : {patient_info}' if patient_info else ''
    prompt = f"""<|system|>\n{SYSTEM_PROMPT}</s>\n<|user|>\nAnalyse ce cas clinique et fournis un diagnostic complet avec recommandations.{patient_str}\nSymptômes : {symptoms}</s>\n<|assistant|>\n"""
    
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=512
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.15
        )
    
    # Extraction de la réponse uniquement (sans le prompt)
    input_len = inputs['input_ids'].shape[1]
    response_ids = outputs[0][input_len:]
    response = tokenizer.decode(response_ids, skip_special_tokens=True)
    
    return response.strip()

print('✅ Fonction diagnose() prête')

In [ ]:
# ============================================================
# CELLULE 3 — Tests sur de nouveaux cas (non vus en entraînement)
# ============================================================

test_cases = [
    {
        'patient': 'Femme 32 ans, enceinte 28 semaines',
        'symptoms': 'Douleurs abdominales basses, contractions irrégulières, pertes vaginales anormales, fièvre légère 38.2°C'
    },
    {
        'patient': 'Enfant 5 ans',
        'symptoms': 'Fièvre 39.5°C, éruption cutanée rosée généralisée, toux, yeux rouges, signe de Koplik visible'
    },
    {
        'patient': 'Homme 67 ans, tabagique chronique',
        'symptoms': 'Dyspnée progressive d\'effort depuis 6 mois, toux chronique productive, sifflements respiratoires, VEMS/CV < 0.70 à la spirométrie'
    }
]

for i, case in enumerate(test_cases, 1):
    print(f'\n{'='*60}')
    print(f'CAS TEST #{i}')
    print(f'Patient  : {case["patient"]}')
    print(f'Symptômes: {case["symptoms"]}')
    print(f'{'='*60}')
    result = diagnose(case['symptoms'], case['patient'])
    print(result)
    print()

In [ ]:
# ============================================================
# CELLULE 4 — Évaluation quantitative (ROUGE Score)
# ============================================================
import evaluate
from datasets import load_from_disk

rouge = evaluate.load('rouge')

dataset = load_from_disk('/content/medical_ai/data/medical_dataset')
val_data = dataset['validation'].select(range(min(20, len(dataset['validation']))))

print(f'⏳ Évaluation sur {len(val_data)} exemples de validation...')

predictions = []
references  = []

for example in val_data:
    # Extraire l'instruction du texte formaté
    instruction = example['instruction']
    reference   = example['response']
    
    # Générer la prédiction
    pred = diagnose(instruction, max_new_tokens=300)
    predictions.append(pred)
    references.append(reference)

results = rouge.compute(predictions=predictions, references=references)

print(f'\n📊 RÉSULTATS D\'ÉVALUATION')
print(f'   ROUGE-1  : {results["rouge1"]:.4f} (précision des mots)')
print(f'   ROUGE-2  : {results["rouge2"]:.4f} (précision des bigrammes)')
print(f'   ROUGE-L  : {results["rougeL"]:.4f} (plus longue sous-séquence)')
print(f'\n   Un score ROUGE-1 > 0.40 est considéré bon pour ce type de tâche.')

In [ ]:
# ============================================================
# CELLULE 5 — Interface Gradio interactive
# ============================================================
import gradio as gr

def gradio_diagnose(symptoms, patient_info, temperature, max_tokens):
    if not symptoms.strip():
        return '⚠️ Veuillez décrire les symptômes du patient.'
    result = diagnose(
        symptoms=symptoms,
        patient_info=patient_info,
        max_new_tokens=int(max_tokens),
        temperature=float(temperature)
    )
    return result

# Exemples prédéfinis
examples = [
    ['Fièvre 39°C, toux productive, douleurs thoraciques, dyspnée depuis 4 jours',
     'Homme 50 ans, fumeur', 0.3, 600],
    ['Douleur thoracique irradiant au bras gauche, sueurs froides, nausées',
     'Homme 60 ans, hypertendu', 0.3, 600],
    ['Polyurie, polydipsie, amaigrissement rapide, fatigue extrême',
     'Adolescent 15 ans', 0.3, 600],
    ['Céphalées sévères, raideur de la nuque, fièvre 40°C, photophobie',
     'Jeune adulte 20 ans', 0.3, 600]
]

with gr.Blocks(
    title='MediAI — Diagnostic Assistant',
    theme=gr.themes.Soft(primary_hue='blue')
) as demo:
    
    gr.Markdown("""
    # 🏥 MediAI — Assistant de Diagnostic Médical
    **Prototype basé sur BioMedLM fine-tuné**
    
    > ⚠️ **Avertissement** : Cet outil est un prototype de recherche. Il ne remplace pas un avis médical professionnel.
    > En cas d'urgence, appelez le **15 (SAMU)** ou le **112**.
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown('### 📝 Saisie du cas clinique')
            symptoms_input = gr.Textbox(
                label='Symptômes *',
                placeholder='Décrivez les symptômes : fièvre, douleurs, durée, intensité...',
                lines=4
            )
            patient_input = gr.Textbox(
                label='Informations patient (optionnel)',
                placeholder='Ex: Homme 45 ans, diabétique, antécédents cardiovasculaires',
                lines=2
            )
            with gr.Accordion('⚙️ Paramètres avancés', open=False):
                temp_slider = gr.Slider(
                    minimum=0.1, maximum=0.9, value=0.3, step=0.1,
                    label='Température (0.1=conservateur, 0.9=créatif)'
                )
                tokens_slider = gr.Slider(
                    minimum=200, maximum=800, value=600, step=100,
                    label='Longueur max de la réponse'
                )
            btn = gr.Button('🔍 Analyser le cas clinique', variant='primary', size='lg')
        
        with gr.Column(scale=1):
            gr.Markdown('### 📋 Analyse et Diagnostic')
            output = gr.Markdown(label='Résultat')
    
    gr.Markdown('### 💡 Exemples de cas cliniques')
    gr.Examples(
        examples=examples,
        inputs=[symptoms_input, patient_input, temp_slider, tokens_slider],
        label='Cliquez pour charger un exemple'
    )
    
    btn.click(
        fn=gradio_diagnose,
        inputs=[symptoms_input, patient_input, temp_slider, tokens_slider],
        outputs=output
    )

print('🚀 Lancement de l\'interface...')
demo.launch(
    share=True,          # Génère un lien public temporaire
    debug=False,
    show_error=True
)